In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded successfully')

In [ ]:
monthly = pd.read_csv('data/processed/monthly_aggregated.csv', parse_dates=['ds'])
monthly = monthly.sort_values('ds').reset_index(drop=True)

print(f'Loaded {len(monthly)} months of data')
print(f'Date range: {monthly["ds"].min().date()} to {monthly["ds"].max().date()}')
monthly[['ds', 'total_qty', 'total_revenue']].tail(5)

### Creating Prophet DataFrames

Prophet requires exactly two columns:
- `ds` — the date column (already named correctly)
- `y`  — the target variable

We create two separate dataframes — one for quantity, one for revenue.

In [ ]:
# Quantity dataframe
df_qty = monthly[['ds', 'total_qty']].rename(columns={'total_qty': 'y'})

# Revenue dataframe
df_rev = monthly[['ds', 'total_revenue']].rename(columns={'total_revenue': 'y'})

print('Quantity dataframe:')
print(df_qty.tail(3).to_string(index=False))
print()
print('Revenue dataframe:')
print(df_rev.tail(3).to_string(index=False))

### Train Test Split

We hold out the last 3 months (Oct–Dec 2025) as a test set to evaluate model accuracy before forecasting Jan 2026.

- **Train:** Jan 2023 – Sep 2025 (33 months)
- **Test:**  Oct 2025 – Dec 2025 (3 months)
- **Forecast target:** Jan 2026

In [ ]:
SPLIT_DATE = '2025-09-30'

# Quantity split
train_qty = df_qty[df_qty['ds'] <= SPLIT_DATE].copy()
test_qty  = df_qty[df_qty['ds'] >  SPLIT_DATE].copy()

# Revenue split
train_rev = df_rev[df_rev['ds'] <= SPLIT_DATE].copy()
test_rev  = df_rev[df_rev['ds'] >  SPLIT_DATE].copy()

print(f'Train size : {len(train_qty)} months ({train_qty["ds"].min().date()} to {train_qty["ds"].max().date()})')
print(f'Test size  : {len(test_qty)} months  ({test_qty["ds"].min().date()} to {test_qty["ds"].max().date()})')

### Build & Train PROPHET Model for Quantity

**Key Prophet parameters explained:**

| Parameter | Value | Reason |
|---|---|---|
| `seasonality_mode` | additive | Seasonal amplitude is constant (from decomposition) |
| `changepoint_prior_scale` | 0.3 | Higher value = more flexible trend; captures Jun 2023 structural break |
| `seasonality_prior_scale` | 10 | Allows stronger seasonal patterns |
| `yearly_seasonality` | True | 12-month cycle present in data |
| `weekly_seasonality` | False | Monthly data — no weekly pattern |
| `daily_seasonality` | False | Monthly data — no daily pattern |
| `changepoints` | Jun 2023 | Manually specify the ramp-up structural break |

In [ ]:
# Build Prophet model for Quantity
model_qty = Prophet(
    seasonality_mode       = 'additive',
    changepoint_prior_scale= 0.3,
    seasonality_prior_scale= 10,
    yearly_seasonality     = True,
    weekly_seasonality     = False,
    daily_seasonality      = False,
    changepoints           = ['2023-06-01']   # Known structural break
)

# Fit on training data
model_qty.fit(train_qty)
print('Prophet Quantity model trained successfully')

### Build & Train PROPHET Mode for Revenue

In [ ]:
model_rev = Prophet(
    seasonality_mode       = 'additive',
    changepoint_prior_scale= 0.3,
    seasonality_prior_scale= 10,
    yearly_seasonality     = True,
    weekly_seasonality     = False,
    daily_seasonality      = False,
    changepoints           = ['2023-06-01']
)

model_rev.fit(train_rev)
print('Prophet Revenue model trained successfully')

### Generate Forecast

We forecast 4 months ahead (Oct, Nov, Dec 2025 + Jan 2026).
- Oct–Dec 2025 = test set evaluation
- Jan 2026 = our actual prediction target

In [ ]:
# Create future dataframe — 4 months ahead from end of training
future_qty = model_qty.make_future_dataframe(periods=4, freq='MS')
future_rev = model_rev.make_future_dataframe(periods=4, freq='MS')

# Generate forecasts
forecast_qty = model_qty.predict(future_qty)
forecast_rev = model_rev.predict(future_rev)

# Extract Jan 2026 prediction
jan2026_qty = forecast_qty[forecast_qty['ds'] == '2026-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
jan2026_rev = forecast_rev[forecast_rev['ds'] == '2026-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']]

print('=== PROPHET FORECAST: JANUARY 2026 ===')
print()
print('Quantity (Units):')
print(f'  Predicted : {jan2026_qty["yhat"].values[0]:,.0f} units')
print(f'  Lower     : {jan2026_qty["yhat_lower"].values[0]:,.0f} units')
print(f'  Upper     : {jan2026_qty["yhat_upper"].values[0]:,.0f} units')
print()
print('Revenue (LAK):')
print(f'  Predicted : LAK {jan2026_rev["yhat"].values[0]:,.0f}')
print(f'  Lower     : LAK {jan2026_rev["yhat_lower"].values[0]:,.0f}')
print(f'  Upper     : LAK {jan2026_rev["yhat_upper"].values[0]:,.0f}')

### Evaluate on Test Set (Oct - Dec 2025)

In [ ]:
def evaluate_prophet(forecast, test_df, label):
    # Merge forecast with actuals
    eval_df = test_df.merge(
        forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']],
        on='ds'
    )

    mae  = mean_absolute_error(eval_df['y'], eval_df['yhat'])
    rmse = np.sqrt(mean_squared_error(eval_df['y'], eval_df['yhat']))
    mape = (np.abs((eval_df['y'] - eval_df['yhat']) / eval_df['y'])).mean() * 100

    print(f'=== {label} — Test Set Evaluation (Oct-Dec 2025) ===')
    print(f'MAE  : {mae:,.2f}')
    print(f'RMSE : {rmse:,.2f}')
    print(f'MAPE : {mape:.2f}%')
    print()
    print('Month-by-month comparison:')
    for _, row in eval_df.iterrows():
        err = ((row['yhat'] - row['y']) / row['y']) * 100
        print(f"  {row['ds'].strftime('%b %Y')}: Actual={row['y']:,.0f} | Predicted={row['yhat']:,.0f} | Error={err:+.1f}%")
    print()
    return mae, rmse, mape

mae_qty, rmse_qty, mape_qty = evaluate_prophet(forecast_qty, test_qty, 'QUANTITY')
mae_rev, rmse_rev, mape_rev = evaluate_prophet(forecast_rev, test_rev, 'REVENUE')

### Plot FORECAST for Quantity

In [ ]:
import os
os.makedirs('reports', exist_ok=True)

fig, ax = plt.subplots(figsize=(16, 6))

# Plot actuals
ax.plot(df_qty['ds'], df_qty['y'], color='steelblue', linewidth=2,
        marker='o', markersize=5, label='Actual', zorder=3)

# Plot forecast line
ax.plot(forecast_qty['ds'], forecast_qty['yhat'], color='darkorange',
        linewidth=2, linestyle='--', label='Prophet Forecast', zorder=3)

# Confidence interval
ax.fill_between(forecast_qty['ds'], forecast_qty['yhat_lower'],
                forecast_qty['yhat_upper'], alpha=0.15, color='darkorange',
                label='Confidence Interval')

# Highlight Jan 2026 prediction
jan_pred = forecast_qty[forecast_qty['ds'] == '2026-01-01']
ax.scatter(jan_pred['ds'], jan_pred['yhat'], color='red', s=120,
           zorder=5, label=f'Jan 2026 Forecast: {jan_pred["yhat"].values[0]:,.0f} units')

# Vertical line at train/test split
ax.axvline(pd.Timestamp('2025-10-01'), color='gray', linestyle=':', linewidth=1.5)
ax.text(pd.Timestamp('2025-10-05'), df_qty['y'].max()*0.92, 'Test →', fontsize=9, color='gray')

ax.set_title('Prophet Forecast — Monthly Quantity Shipped (with Jan 2026 Prediction)', fontweight='bold', fontsize=13)
ax.set_ylabel('Quantity (Units)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('reports/08_prophet_qty_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/08_prophet_qty_forecast.png')

### Plot FORECAST for Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df_rev['ds'], df_rev['y'], color='steelblue', linewidth=2,
        marker='o', markersize=5, label='Actual', zorder=3)
ax.plot(forecast_rev['ds'], forecast_rev['yhat'], color='darkorange',
        linewidth=2, linestyle='--', label='Prophet Forecast', zorder=3)
ax.fill_between(forecast_rev['ds'], forecast_rev['yhat_lower'],
                forecast_rev['yhat_upper'], alpha=0.15, color='darkorange',
                label='Confidence Interval')

jan_pred_rev = forecast_rev[forecast_rev['ds'] == '2026-01-01']
ax.scatter(jan_pred_rev['ds'], jan_pred_rev['yhat'], color='red', s=120,
           zorder=5, label=f'Jan 2026 Forecast: LAK {jan_pred_rev["yhat"].values[0]/1e9:,.1f}B')

ax.axvline(pd.Timestamp('2025-10-01'), color='gray', linestyle=':', linewidth=1.5)
ax.text(pd.Timestamp('2025-10-05'), df_rev['y'].max()*0.92, 'Test →', fontsize=9, color='gray')

ax.set_title('Prophet Forecast — Monthly Revenue LAK (with Jan 2026 Prediction)', fontweight='bold', fontsize=13)
ax.set_ylabel('Revenue (LAK)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:,.1f}B'))
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig('reports/09_prophet_rev_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/09_prophet_rev_forecast.png')

### Plot Prophet Components

In [ ]:
fig = model_qty.plot_components(forecast_qty)
fig.suptitle('Prophet Components — Quantity Model', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('reports/10_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/10_prophet_components.png')

### Full Summary and Save Results

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# Save full forecast
forecast_qty[['ds','yhat','yhat_lower','yhat_upper']].to_csv('data/processed/prophet_forecast_qty.csv', index=False)
forecast_rev[['ds','yhat','yhat_lower','yhat_upper']].to_csv('data/processed/prophet_forecast_rev.csv', index=False)

print('Forecasts saved.')
print()

# Final summary
print('=' * 55)
print('   PROPHET MODEL SUMMARY')
print('=' * 55)
print()
print('MODEL CONFIGURATION:')
print('  Mode              : Additive')
print('  Changepoint Scale : 0.3')
print('  Manual Changepoint: Jun 2023 (ramp-up end)')
print('  Yearly Seasonality: True')
print()
print('TEST SET PERFORMANCE (Oct-Dec 2025):')
print(f'  Quantity MAPE : {mape_qty:.2f}%')
print(f'  Revenue  MAPE : {mape_rev:.2f}%')
print()
print('JANUARY 2026 FORECAST:')
print(f'  Quantity : {jan2026_qty["yhat"].values[0]:,.0f} units')
print(f'             (range: {jan2026_qty["yhat_lower"].values[0]:,.0f} to {jan2026_qty["yhat_upper"].values[0]:,.0f})')
print(f'  Revenue  : LAK {jan2026_rev["yhat"].values[0]:,.0f}')
print(f'             (range: LAK {jan2026_rev["yhat_lower"].values[0]:,.0f} to {jan2026_rev["yhat_upper"].values[0]:,.0f})')